In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##Packages

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

from statsmodels.tsa.arima.model import ARIMA

from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

#Eurostat

##Load dataset

In [ ]:
#Eurostat import export data
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Thesis_ESCP/Eurostat_dataset.csv', sep=';')
display(df.head())

In [ ]:
#nb of rows and columns of dataset df
print(df.shape)

##Data Cleaning

In [ ]:
# Drop specified columns from the dataset
df = df.drop(columns=['DATAFLOW', 'LAST UPDATE', 'freq', 'reporter', 'partner'])

# Change 'TIME_PERIOD' to 'Month'
df = df.rename(columns={'TIME_PERIOD': 'Month'})

# Convert 'Month' column in df to datetime
df['Month'] = pd.to_datetime(df['Month'], errors='coerce')

#add '_' between spaces for unique values of product
df['product'] = df['product'].str.replace(' ', '_')

#change product value TOTAL to Total
df['product'] = df['product'].replace('TOTAL', 'Total')

In [ ]:
#product unique values
print(df['product'].unique())

In [ ]:
#only keep product = 'Food_and_live_animals', 'Non-ferrous_metals', 'Iron_and_steel'
df = df[df['product'].isin(['Food_and_live_animals', 'Non-ferrous_metals', 'Iron_and_steel','Crude_materials,_inedible,_except_fuels',
                            #'Total'#
                            ])]

In [ ]:
display(df.head())

##Split import and export

In [ ]:
# Create two separate DataFrames
df_export = df[df['flow'] == 'EXPORT'].copy()
df_import = df[df['flow'] == 'IMPORT'].copy()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x=df_export['OBS_VALUE'])
plt.title("Boxplot of OBS_VALUE (Exports)")
plt.show()

plt.figure(figsize=(10,5))
sns.histplot(df_export['OBS_VALUE'], bins=50, kde=True)
plt.title("Distribution of OBS_VALUE (Exports)")
plt.show()

In [ ]:
sns.boxplot(x=df_import['OBS_VALUE'])
plt.title("Boxplot of OBS_VALUE (Imports)")
plt.show()

plt.figure(figsize=(10,5))
sns.histplot(df_export['OBS_VALUE'], bins=50, kde=True)
plt.title("Distribution of OBS_VALUE (Exports)")
plt.show()

##Log OBS_VALUE (Target value)

In [ ]:
# For exports
df_export["log_OBS_VALUE"] = np.log1p(df_export["OBS_VALUE"])

# For imports
df_import["log_OBS_VALUE"] = np.log1p(df_import["OBS_VALUE"])


In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x=df_export['log_OBS_VALUE'])
plt.title("Boxplot of OBS_VALUE (Exports)")
plt.show()

plt.figure(figsize=(10,5))
sns.histplot(df_export['log_OBS_VALUE'], bins=50, kde=True)
plt.title("Distribution of OBS_VALUE (Exports)")
plt.show()

In [ ]:
df_export.loc[df_export['log_OBS_VALUE'] == df_export['log_OBS_VALUE'].max()]

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x=df_import['log_OBS_VALUE'])
plt.title("Boxplot of OBS_VALUE (Imports)")
plt.show()

plt.figure(figsize=(10,5))
sns.histplot(df_import['log_OBS_VALUE'], bins=50, kde=True)
plt.title("Distribution of OBS_VALUE (Imports)")
plt.show()

#Exchange Rate

In [ ]:
df_rate = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Thesis_ESCP/ECB Data Portal_20251019064740.csv', sep=',', header=0)
print(df_rate)

##Data Cleaning

In [ ]:
df_rate.rename(columns={'DATE': 'Month', 'US dollar/Euro (EXR.M.USD.EUR.SP00.A)': 'EUR_USD'}, inplace=True)
df_rate['Month'] = pd.to_datetime(df_rate['Month'])
df_rate= df_rate.drop(columns=['TIME PERIOD'])

In [ ]:
#change Month so that it always starts at the start of the month 01
df_rate['Month'] = df_rate['Month'].dt.strftime('%Y-%m-01')
#drop Month before 2002
df_rate = df_rate[df_rate['Month'] >= '2002-01-01']

In [ ]:
print(df_rate)

In [ ]:
# Ensure 'Month' is in datetime format
df_rate['Month'] = pd.to_datetime(df_rate['Month'], errors='coerce')

# Plot the EUR/USD line chart
plt.figure(figsize=(10, 5))
plt.plot(df_rate['Month'], df_rate['EUR_USD'], color='steelblue', linewidth=2)

# Titles and labels
#plt.title('Monthly EUR/USD Exchange Rate (2002–2025)', fontsize=13, fontweight='bold')
plt.xlabel('Year', fontsize=11)
plt.ylabel('EUR/USD Exchange Rate', fontsize=11)

# Format x-axis to show only the year (e.g., 2002, 2003, ...)
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45)

# Add grid and adjust layout
#plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()

# Save figure (optional)
plt.savefig('EUR_USD_Exchange_Rate_LineChart.png', dpi=300)

# Show plot
plt.show()

# COVID policy indicators and indices

##Load dataset

In [ ]:
df_covid = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Thesis_ESCP/OxCGRT_simplified_v1.csv', sep=';', header=0)
print(df_covid) # Displays the entire DataFrame

##Data Cleaning

In [ ]:
#Keep only relevant columns
df_covid = df_covid[["CountryName", "CountryCode", "Date", "StringencyIndex_Average"]]

# Filter for EU27 countries
eu_countries = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary",
    "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta",

    "Netherlands", "Poland", "Portugal", "Romania", "Slovakia", "Slovenia",
    "Spain", "Sweden"
]
df_covid = df_covid[df_covid["CountryName"].isin(eu_countries)]

# Convert date column to datetime format
df_covid["Date"] = pd.to_datetime(df_covid["Date"], format="%Y%m%d", errors="coerce")

# Create a 'Month' column (YYYY-MM)
df_covid["Month"] = df_covid["Date"].dt.to_period("M")

# Convert 'StringencyIndex_Average' to numeric, coercing errors
df_covid['StringencyIndex_Average'] = pd.to_numeric(df_covid['StringencyIndex_Average'], errors='coerce')

#Compute monthly averages per country
monthly_eu = (
    df_covid.groupby(["CountryName", "Month"])["StringencyIndex_Average"]
    .mean()
    .reset_index()
)

monthly_eu.rename(columns={"StringencyIndex_Average": "EU_Stringency_Index"}, inplace=True)

#Convert Month back to YYYY-MM format
monthly_eu["Month"] = monthly_eu["Month"].astype(str)

#Save the EU monthly averages to a CSV file
monthly_eu.to_csv("EU_COVID_Stringency_Monthly.csv", index=False)

print(monthly_eu.head(12))  # preview the first 12 months

In [ ]:
# Add COVID_Dummy column (1 during 2020–2021, 0 otherwise)
monthly_eu["COVID_Dummy"] = monthly_eu["Month"].apply(
    lambda x: 1 if x[:4] in ["2020", "2021"] else 0
)

In [ ]:
print(monthly_eu.head())

In [ ]:
#number of rows and columns
print(monthly_eu.shape)

In [ ]:
# Group by Month and plot average COVID_Dummy
ax = monthly_eu.groupby("Month")["COVID_Dummy"].mean().plot(
    kind="bar",
    figsize=(10, 5),
    color="steelblue",
    edgecolor="black"
)

# Convert 'Month' column to datetime objects before formatting
monthly_eu["Month_dt"] = pd.to_datetime(monthly_eu["Month"])

# Format x-axis to show YYYY-MM instead of full datetime
ax.set_xticklabels(
    [d.strftime('%Y-%m') for d in monthly_eu["Month_dt"].unique()],
    rotation=45,
    ha='right'
)

# Titles and labels
plt.title("COVID_Dummy Variable by Month (1 = Pandemic Period, 0 = Non-Pandemic)", fontsize=13, fontweight='bold')
plt.xlabel("Month (YYYY-MM)")
plt.ylabel("COVID_Dummy")

plt.tight_layout()
plt.show()

#Commodity Prices

In [ ]:
df_price = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Thesis_ESCP/Worldbank_commodityPrices_Monthly.xlsx',
                         sheet_name='Monthly Prices',
                         header=2)

# Display the dataframe
print(df_price.head())

##Data Cleaning

In [ ]:
#change column name Unnamed: 0 to MOnthly
df_price = df_price.rename(columns={'Unnamed: 0': 'Month'})

In [ ]:
for col in df_price.columns:
    if col != 'Month':
        df_price[col] = pd.to_numeric(df_price[col], errors='coerce')

print(df_price.dtypes)

In [ ]:
#change Month column format to yyyy-MM
# Step 1: Replace 'M' with '-' and ensure consistent format
df_price['Month'] = df_price['Month'].str.replace('M', '-', regex=False)

# Step 2: Convert to datetime
df_price['Month'] = pd.to_datetime(df_price['Month'], errors='coerce')

# Step 3: (Optional) format as yyyy-MM string (if you want it as text, not datetime)
#df_price['Month'] = df_price['Month'].dt.strftime('%Y-%m')

#only have data from 2003-01-01
df_price = df_price[df_price['Month'] >= '2003-01-01']

In [ ]:
print(df_price.head())

In [ ]:
category_groups = {
    "Food_and_live_animals": [
        "COCOA", "COFFEE_ARABIC", "COFFEE_ROBUS", "TEA_AVG", "TEA_COLOMBO", "TEA_KOLKATA", "TEA_MOMBASA",
        "COCONUT_OIL", "PALM_OIL", "PLMKRNL_OIL", "SOYBEANS", "SOYBEAN_OIL", "SOYBEAN_MEAL",
        "RAPESEED_OIL", "SUNFLOWER_OIL", "MAIZE", "RICE_05", "RICE_25", "RICE_A1",
        "WHEAT_US_SRW", "WHEAT_US_HRW", "SUGAR_EU", "SUGAR_US", "SUGAR_WLD", "BEEF", "CHICKEN", "LAMB"
    ],
    "Iron_and_steel": ["IRON_ORE"],
    "Non_ferrous_metals": ["ALUMINUM", "COPPER", "LEAD", "Tin", "NICKEL", "Zinc"]
}

df_price['Month'] = pd.to_datetime(df_price['Month'], errors='coerce')

category_indices = pd.DataFrame({'Month': df_price['Month']})

for category, cols in category_groups.items():
    valid_cols = [c for c in cols if c in df_price.columns]
    category_indices[category] = df_price[valid_cols].mean(axis=1, skipna=True)

print(category_indices.head())

# Reshape to long format
category_indices_long = category_indices.melt(
    id_vars=['Month'],
    var_name='product',
    value_name='Price_Index'
)

# Optional: clean & sort
category_indices_long = category_indices_long.sort_values(['product', 'Month']).reset_index(drop=True)

print(category_indices_long.head(10))

###convert Price index from US dollards to EUR

In [ ]:
# Ensure Month columns are datetime
df_rate['Month'] = pd.to_datetime(df_rate['Month'], errors='coerce')
df_price['Month'] = pd.to_datetime(df_price['Month'], errors='coerce')
# Merge ECB exchange rate into the price dataset
category_indices_long = pd.merge(category_indices_long, df_rate[['Month', 'EUR_USD']], on='Month', how='left')
# Convert USD-based price index to EUR-based
category_indices_long['Price_Index_EUR'] = category_indices_long['Price_Index'] / category_indices_long['EUR_USD']
#drop Price_Index ($)
category_indices_long = category_indices_long.drop(columns=['Price_Index'])
#chnage name price index euro to price index
category_indices_long.rename(columns={'Price_Index_EUR': 'Price_Index'}, inplace=True)

In [ ]:
print(category_indices_long.head())

In [ ]:
plt.figure(figsize=(10,5))
for cat in ['Food_and_live_animals', 'Iron_and_steel', 'Non_ferrous_metals']:
    subset = category_indices_long[category_indices_long['product'] == cat]
    plt.plot(subset['Month'], subset['Price_Index'], label=cat)

plt.title('Monthly Commodity Price Indices in EUR (2003–2025)', fontsize=13, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Price Index (EUR)')
plt.legend()
plt.tight_layout()
plt.show()


#EU GDP

In [ ]:
df_gdp = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Thesis_ESCP/GDP_EU.xlsx',
                         sheet_name='Sheet 1',
                         header=0)

# Display the dataframe
print(df_gdp.head())

##Data Cleaning

In [ ]:
#keep only where row Geo(Labels) = European Union - 27 countries (from 2020)
df_gdp = df_gdp[df_gdp['TIME'] == 'European Union - 27 countries (from 2020)']
#delete col where its Unnamed:
df_gdp = df_gdp.loc[:, ~df_gdp.columns.str.contains('^Unnamed')]

In [ ]:
print(df_gdp.head())

In [ ]:
# Transpose so that quarters become rows
df_gdp = df_gdp.set_index('TIME').T
df_gdp = df_gdp.reset_index()
df_gdp.columns = ['Quarter', 'GDP_EU']

# --- Convert quarter labels to proper datetime ---
df_gdp['Quarter'] = df_gdp['Quarter'].str.strip()
df_gdp = df_gdp[df_gdp['Quarter'].str.match(r'^\d{4}-Q[1-4]$')]
df_gdp['Month'] = pd.PeriodIndex(df_gdp['Quarter'], freq='Q').to_timestamp(how='end')

# --- Convert GDP values to numeric ---
df_gdp['GDP_EU'] = pd.to_numeric(df_gdp['GDP_EU'], errors='coerce')

# Check quarterly data
print("Quarterly GDP format:")
print(df_gdp.head())

# --- Convert quarterly → monthly ---
# Set datetime as index and ensure sorted order
df_gdp = df_gdp.set_index('Month').sort_index()

# Resample to monthly ('MS' = month start)
df_gdp_monthly = df_gdp.resample('MS').interpolate(method='linear')

# Reset index for merging
df_gdp_monthly = df_gdp_monthly.reset_index()[['Month', 'GDP_EU']]

print("\n Monthly GDP (interpolated):")
print(df_gdp_monthly.head(15))

In [ ]:
# Create a mapping from quarter to its months
quarter_months = {
    'Q1': ['01', '02', '03'],
    'Q2': ['04', '05', '06'],
    'Q3': ['07', '08', '09'],
    'Q4': ['10', '11', '12']
}

# Create an expanded dataframe
expanded_rows = []

for _, row in df_gdp.iterrows():
    year, q = row['Quarter'].split('-')
    for m in quarter_months[q]:
        month_str = f"{year}-{m}-01"
        expanded_rows.append({'Month': pd.to_datetime(month_str), 'GDP_EU': row['GDP_EU']})

df_gdp_monthly = pd.DataFrame(expanded_rows).sort_values('Month').reset_index(drop=True)

# Check result
print(df_gdp_monthly.head(12))

In [ ]:
#nb of row and column
print(df_gdp_monthly.shape)

#Merge Datasets

In [ ]:
# Ensure Month columns are datetime
df_export['Month'] = pd.to_datetime(df_export['Month'], errors='coerce')
df_import['Month'] = pd.to_datetime(df_import['Month'], errors='coerce')
category_indices_long['Month'] = pd.to_datetime(category_indices_long['Month'], errors='coerce')

##Export

In [ ]:
# Merge commodity price indices into export data
df_export_merged1 = pd.merge(
    df_export,
    category_indices_long[['Month', 'product', 'Price_Index']],
    on=['Month', 'product'],
    how='left'
)

# Drop rows without price data
df_export_merged1 = df_export_merged1.dropna(subset=['Price_Index'])

print(df_export_merged1.head())

In [ ]:
# Ensure Month columns are datetime in all dataframes
df_export_merged1['Month'] = pd.to_datetime(df_export_merged1['Month'], errors='coerce')
monthly_eu['Month'] = pd.to_datetime(monthly_eu['Month'], errors='coerce')


# --- Merge macro data (COVID_Dummy) into Export dataset ---
df_export_merged2 = pd.merge(
    df_export_merged1,
    monthly_eu[["Month", "COVID_Dummy"]],
    on="Month",
    how="left"
)

# Reorder columns so 'Month' and 'COVID_Dummy' come first
cols_to_remove = ["Month", "COVID_Dummy"]
if "EU_Stringency_Index" in df_export_merged2.columns:
    cols_to_remove.append("EU_Stringency_Index")

df_export_merged2 = df_export_merged2[
    ["Month", "COVID_Dummy"] + [col for col in df_export_merged2.columns if col not in cols_to_remove]
]

print(df_export_merged2.head())

In [ ]:
# --- EXPORT DATASET ---
# Sort by product and Month (no need for 'flow' since exports are separate)
df_export_merged2 = df_export_merged2.sort_values(["product", "Month"]).reset_index(drop=True)

# Fill missing COVID_Dummy values with 0
df_export_merged2["COVID_Dummy"] = df_export_merged2["COVID_Dummy"].fillna(0)

# Rename for consistency
df_export_final = df_export_merged2.copy()

print(df_export_final.head())

In [ ]:
df_export_final = pd.merge(df_export_final, df_gdp_monthly, on='Month', how='left')

In [ ]:
#drop where GDP_EU missing value
df_export_final = df_export_final.dropna(subset=['GDP_EU'])

In [ ]:
#  Check missing values for EXPORT dataset
print(df_export_final.isnull().sum())

In [ ]:
#  Correlation for EXPORT dataset
corr_export = df_export_final.corr(numeric_only=True)
print(corr_export)

In [ ]:
import seaborn as sns
corr = df_export_final[['Price_Index','GDP_EU','log_OBS_VALUE']].corr()
plt.figure(figsize=(5,4))
sns.heatmap(corr, annot=True, cmap='Blues', fmt=".2f")
plt.title('Correlation between Commodity Prices, GDP, and Trade Values', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
#drop COVID_Dummy
df_export_final = df_export_final.drop(columns=['COVID_Dummy'])

In [ ]:
#count of product categories
print(df_export_final['product'].value_counts())

##Import

In [ ]:
# --- Merge commodity price indices into import data ---
df_import_merged1 = pd.merge(
    df_import,
    category_indices_long[['Month', 'product', 'Price_Index']],
    on=['Month', 'product'],
    how='left'
)

# Drop rows without price data
df_import_merged1 = df_import_merged1.dropna(subset=['Price_Index'])

print(df_import_merged1.head())

In [ ]:
# Ensure Month columns are datetime in all dataframes
df_export_merged1['Month'] = pd.to_datetime(df_export_merged1['Month'], errors='coerce')
df_import_merged1['Month'] = pd.to_datetime(df_import_merged1['Month'], errors='coerce')
monthly_eu['Month'] = pd.to_datetime(monthly_eu['Month'], errors='coerce')

# --- Merge macro data (COVID_Dummy) into Import dataset ---
df_import_merged2 = pd.merge(
    df_import_merged1,
    monthly_eu[["Month", "COVID_Dummy"]],
    on="Month",
    how="left"
)

# Reorder columns so 'Month' and 'COVID_Dummy' come first
cols_to_remove = ["Month", "COVID_Dummy"]
if "EU_Stringency_Index" in df_import_merged2.columns:
    cols_to_remove.append("EU_Stringency_Index")

df_import_merged2 = df_import_merged2[
    ["Month", "COVID_Dummy"] + [col for col in df_import_merged2.columns if col not in cols_to_remove]
]

print(df_import_merged2.head())

In [ ]:
# --- IMPORT DATASET ---
# Sort by product and Month
df_import_merged2 = df_import_merged2.sort_values(["product", "Month"]).reset_index(drop=True)

# Fill missing COVID_Dummy values with 0
df_import_merged2["COVID_Dummy"] = df_import_merged2["COVID_Dummy"].fillna(0)

# Rename for consistency (if you want a unified naming convention)
df_import_final = df_import_merged2.copy()

print(df_import_final.head())

In [ ]:
df_import_final = pd.merge(df_import_final, df_gdp_monthly, on='Month', how='left')

In [ ]:
df_import_final = df_import_final.dropna(subset=['GDP_EU'])

In [ ]:
#  Check missing values for IMPORT dataset
print(df_import_final.isnull().sum())

In [ ]:
#Correlation for IMPORT dataset
corr_import = df_import_final.corr(numeric_only=True)
print(corr_import)

In [ ]:
#drop COVID_Dummy
df_import_final = df_import_final.drop(columns=['COVID_Dummy'])

#Data Analysis

## FEATURE ENGINEERING (Dummies+Lags)

Purpose: create lag features and minimal exogenous variables for forecasting.

In [ ]:
# ADDITIONAL FEATURE CREATION
# ----- EXPORT DATASET -----
df_export_final["month_num"] = df_export_final["Month"].dt.month
df_export_final["year"] = df_export_final["Month"].dt.year
df_export_final["quarter"] = df_export_final["Month"].dt.quarter

# Nonlinear interaction features
df_export_final["Price_GDP_interaction"] = df_export_final["Price_Index"] * df_export_final["GDP_EU"]
df_export_final["Price_change"] = df_export_final.groupby("product")["Price_Index"].pct_change()
df_export_final["GDP_change"] = df_export_final["GDP_EU"].pct_change()

# ----- IMPORT DATASET -----
df_import_final["month_num"] = df_import_final["Month"].dt.month
df_import_final["year"] = df_import_final["Month"].dt.year
df_import_final["quarter"] = df_import_final["Month"].dt.quarter

df_import_final["Price_GDP_interaction"] = df_import_final["Price_Index"] * df_import_final["GDP_EU"]
df_import_final["Price_change"] = df_import_final.groupby("product")["Price_Index"].pct_change()
df_import_final["GDP_change"] = df_import_final["GDP_EU"].pct_change()

In [ ]:
TARGET = "log_OBS_VALUE"
MAX_LAG = 6

def add_lags(block, target=TARGET, max_lag=MAX_LAG):
    """Add lag features for each product (preserves time order)."""
    for L in range(1, max_lag + 1):
        block[f"{target}_lag{L}"] = block[target].shift(L)
    return block

# =============== EXPORT DATASET ===============
df_export_final = df_export_final.sort_values(["product", "Month"])
df_export_model = df_export_final.groupby("product", group_keys=False).apply(add_lags)
lag_cols = [f"{TARGET}_lag{i}" for i in range(1, MAX_LAG + 1)]
df_export_model = df_export_model.dropna(subset=lag_cols)

FEATURES_EXPORT = lag_cols + [
    "Price_Index", "GDP_EU",
    "Price_GDP_interaction", "Price_change", "GDP_change",
    "month_num", "quarter"
]

print(f"✅ Export dataset ready with {len(FEATURES_EXPORT)} features.")
print(df_export_model.head())

# =============== IMPORT DATASET ===============
df_import_final = df_import_final.sort_values(["product", "Month"])
df_import_model = df_import_final.groupby("product", group_keys=False).apply(add_lags)
df_import_model = df_import_model.dropna(subset=lag_cols)

FEATURES_IMPORT = lag_cols + [
    "Price_Index", "GDP_EU",
    "Price_GDP_interaction", "Price_change", "GDP_change",
    "month_num", "quarter"
]

print(f"✅ Import dataset ready with {len(FEATURES_IMPORT)} features.")
print(df_import_model.head())


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Label encode product names for Export and Import
encoder_export = LabelEncoder()
df_export_model["product_encoded"] = encoder_export.fit_transform(df_export_model["product"])

encoder_import = LabelEncoder()
df_import_model["product_encoded"] = encoder_import.fit_transform(df_import_model["product"])

# Add encoded variable to feature list
FEATURES_EXPORT = FEATURES_EXPORT + ["product_encoded"]
FEATURES_IMPORT = FEATURES_IMPORT + ["product_encoded"]

In [ ]:
#count of product categories
print(df_export_model['product'].value_counts())

##MODELS

###1.ARIMA (Econometric Baseline)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
TRAIN_END = "2018-12"
TEST_START = "2019-01"

#### Export

In [ ]:
# =============== EXPORT ARIMAX MODEL ===============
# Aggregate monthly target and exogenous features
series_export = df_export_model.groupby("Month")["log_OBS_VALUE"].sum().asfreq("MS")
X_exog_export = (
    df_export_model.groupby("Month")[["Price_Index", "GDP_EU", "Price_GDP_interaction", "Price_change", "GDP_change"]]
    .mean()
    .asfreq("MS")
    .fillna(method="ffill")
)

# Train-test split
y_train_export = series_export[:TRAIN_END]
y_test_export = series_export[TEST_START:]
X_train_export = X_exog_export[:TRAIN_END]
X_test_export = X_exog_export[TEST_START:]

# Fit SARIMAX model
model_export_arimax = SARIMAX(
    y_train_export, exog=X_train_export,
    order=(1,1,1),
    seasonal_order=(1,1,1,12)
)
model_export_fit = model_export_arimax.fit(disp=False)
print(model_export_fit.summary())

# Forecast on test set
forecast_export = model_export_fit.get_forecast(steps=len(y_test_export), exog=X_test_export)
forecast_mean_export = forecast_export.predicted_mean
forecast_ci_export = forecast_export.conf_int()

# --- Plot forecast ---
plt.figure(figsize=(10,5))
plt.plot(y_train_export, label="Train")
plt.plot(y_test_export, label="Actual", color="blue")
plt.plot(forecast_mean_export, label="Forecast", color="orange")
plt.fill_between(forecast_ci_export.index, forecast_ci_export.iloc[:,0], forecast_ci_export.iloc[:,1], color="orange", alpha=0.2)
plt.title("ARIMAX Forecast – EU Exports")
plt.xlabel("Month")
plt.ylabel("log_OBS_VALUE")
plt.legend()
plt.tight_layout()
plt.show()

# --- Evaluation metrics ---
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

r2_export_arimax = r2_score(y_test_export, forecast_mean_export)
mae_export_arimax = mean_absolute_error(y_test_export, forecast_mean_export)
rmse_export_arimax = np.sqrt(mean_squared_error(y_test_export, forecast_mean_export))

print("\nEXPORT ARIMAX Results:")
print(f"Test R²: {r2_export_arimax:.3f}")
print(f"MAE: {mae_export_arimax:,.2f}")
print(f"RMSE: {rmse_export_arimax:,.2f}")

####Import

In [ ]:
# =============== IMPORT ARIMAX MODEL ===============
# Aggregate monthly averages of exogenous features
series_import = df_import_model.groupby("Month")["log_OBS_VALUE"].sum().asfreq("MS")
X_exog_import = (
    df_import_model.groupby("Month")[["Price_Index", "GDP_EU", "Price_GDP_interaction", "Price_change", "GDP_change"]]
    .mean()
    .asfreq("MS")
    .fillna(method="ffill")
)


# Train-test split
y_train_import = series_import[:TRAIN_END]
y_test_import = series_import[TEST_START:]
X_train_import = X_exog_import[:TRAIN_END]
X_test_import = X_exog_import[TEST_START:]

# Fit ARIMAX model
model_import_arimax = SARIMAX(
    y_train_import, exog=X_train_import,
    order=(1,1,1),
    seasonal_order=(1,1,1,12)
)
model_import_fit = model_import_arimax.fit(disp=False)

print(model_import_fit.summary())

# Forecast
forecast_import = model_import_fit.get_forecast(steps=len(y_test_import), exog=X_test_import)
forecast_mean_import = forecast_import.predicted_mean
forecast_ci_import = forecast_import.conf_int()

# --- Plot forecast ---
plt.figure(figsize=(10,5))
plt.plot(y_train_import, label="Train")
plt.plot(y_test_import, label="Actual", color="blue")
plt.plot(forecast_mean_import, label="Forecast", color="orange")
plt.fill_between(forecast_ci_import.index, forecast_ci_import.iloc[:,0], forecast_ci_import.iloc[:,1], color="orange", alpha=0.2)
plt.title("ARIMAX Forecast – EU Imports")
plt.xlabel("Month")
plt.ylabel("log_OBS_VALUE")
plt.legend()
plt.tight_layout()
plt.show()

# --- Evaluation metrics ---
r2_import_arimax = r2_score(y_test_import, forecast_mean_import)
mae_import_arimax = mean_absolute_error(y_test_import, forecast_mean_import)
rmse_import_arimax = np.sqrt(mean_squared_error(y_test_import, forecast_mean_import))

print("\nIMPORT ARIMAX Results:")
print(f"Test R²: {r2_import_arimax:.3f}")
print(f"MAE: {mae_import_arimax:,.2f}")
print(f"RMSE: {rmse_import_arimax:,.2f}")

###3.Linear Regression (ML Baseline)

####Export

In [ ]:
# =============== EXPORT LINEAR REGRESSION MODEL ===============
# --- Define target and features ---
X_export_lin = df_export_model[FEATURES_EXPORT]
y_export_lin = df_export_model["log_OBS_VALUE"]

# --- Ensure Month is datetime ---
df_export_model["Month"] = pd.to_datetime(df_export_model["Month"], errors='coerce')

# --- TimeSeries Cross-Validation ---
tscv = TimeSeriesSplit(n_splits=5)

model_export_lin = LinearRegression()

cv_r2_scores = cross_val_score(
    model_export_lin,
    X_export_lin,
    y_export_lin,
    cv=tscv,
    scoring="r2"
)

print(f"EXPORT Average CV R²: {cv_r2_scores.mean():.3f}")

# --- Time-based Train-Test Split (for final evaluation) ---
X_train_export_lin = X_export_lin[df_export_model["Month"] <= "2018-12"]
X_test_export_lin  = X_export_lin[df_export_model["Month"] >  "2018-12"]
y_train_export_lin = y_export_lin[df_export_model["Month"] <= "2018-12"]
y_test_export_lin  = y_export_lin[df_export_model["Month"] >  "2018-12"]

# --- Train model on training set ---
model_export_lin.fit(X_train_export_lin, y_train_export_lin)

# --- Predict on test set ---
y_pred_export_lin = model_export_lin.predict(X_test_export_lin)

# --- Evaluate test performance ---
r2_export_lin = r2_score(y_test_export_lin, y_pred_export_lin)
mae_export_lin = mean_absolute_error(y_test_export_lin, y_pred_export_lin)
rmse_export_lin = np.sqrt(mean_squared_error(y_test_export_lin, y_pred_export_lin))

print("\nEXPORT Linear Regression Results:")
print(f"Test R²: {r2_export_lin:.3f}")
print(f"MAE: {mae_export_lin:,.2f}")
print(f"RMSE: {rmse_export_lin:,.2f}")


In [ ]:
# --- Visualize actual vs predicted trade values (in log scale) ---
plt.figure(figsize=(10,5))
plt.plot(
    df_export_model.loc[df_export_model["Month"] > "2018-12", "Month"],
    y_test_export_lin.values,
    label="Actual Export (log scale)"
)
plt.plot(
    df_export_model.loc[df_export_model["Month"] > "2018-12", "Month"],
    y_pred_export_lin,
    label="Predicted Export (Linear Regression, log scale)"
)
plt.title("EU Export Forecast – Linear Regression (log_OBS_VALUE, 2019–2025)")
plt.xlabel("Month")
plt.ylabel("log(Trade Value in EUR)")
plt.legend()
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

####Import

In [ ]:
# =============== IMPORT LINEAR REGRESSION MODEL ===============
# --- Define target and features ---
X_import_lin = df_import_model[FEATURES_IMPORT]
y_import_lin = df_import_model["log_OBS_VALUE"]

# --- Ensure Month is datetime ---
df_import_model["Month"] = pd.to_datetime(df_import_model["Month"], errors='coerce')

# --- TimeSeries Cross-Validation ---
tscv = TimeSeriesSplit(n_splits=5)

model_import_lin = LinearRegression()

cv_r2_scores = cross_val_score(
    model_import_lin,
    X_import_lin,
    y_import_lin,
    cv=tscv,
    scoring="r2"
)

print(f"IMPORT Average CV R²: {cv_r2_scores.mean():.3f}")

# --- Time-based Train-Test Split (for final evaluation) ---
X_train_import_lin = X_import_lin[df_import_model["Month"] <= "2018-12"]
X_test_import_lin  = X_import_lin[df_import_model["Month"] >  "2018-12"]
y_train_import_lin = y_import_lin[df_import_model["Month"] <= "2018-12"]
y_test_import_lin  = y_import_lin[df_import_model["Month"] >  "2018-12"]

# --- Train model on training set ---
model_import_lin.fit(X_train_import_lin, y_train_import_lin)

# --- Predict on test set ---
y_pred_import_lin = model_import_lin.predict(X_test_import_lin)

# --- Evaluate test performance ---
r2_import_lin = r2_score(y_test_import_lin, y_pred_import_lin)
mae_import_lin = mean_absolute_error(y_test_import_lin, y_pred_import_lin)
rmse_import_lin = np.sqrt(mean_squared_error(y_test_import_lin, y_pred_import_lin))

print("\nIMPORT Linear Regression Results:")
print(f"Test R²: {r2_import_lin:.3f}")
print(f"MAE: {mae_import_lin:,.2f}")
print(f"RMSE: {rmse_import_lin:,.2f}")

In [ ]:
# --- Visualize actual vs predicted trade values (in log scale) ---
plt.figure(figsize=(10,5))
plt.plot(
    df_import_model.loc[df_import_model["Month"] > "2018-12", "Month"],
    y_test_import_lin.values,
    label="Actual Import (log scale)"
)
plt.plot(
    df_import_model.loc[df_import_model["Month"] > "2018-12", "Month"],
    y_pred_import_lin,
    label="Predicted Import (Linear Regression, log scale)"
)
plt.title("EU Import Forecast – Linear Regression (log_OBS_VALUE, 2019–2025)")
plt.xlabel("Month")
plt.ylabel("log(Trade Value in EUR)")
plt.legend()
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

###4.Random Forest (Machine Learning)

In [ ]:
# Common config
RANDOM_STATE = 42
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

####Export

In [ ]:
X_export = df_export_model[FEATURES_EXPORT]
y_export = df_export_model["log_OBS_VALUE"]
df_export_model["Month"] = pd.to_datetime(df_export_model["Month"])

X_train_export = X_export[df_export_model["Month"] <= "2018-12"]
X_test_export = X_export[df_export_model["Month"] > "2018-12"]
y_train_export = y_export[df_export_model["Month"] <= "2018-12"]
y_test_export = y_export[df_export_model["Month"] > "2018-12"]

# Tuned Random Forest
rf_model = RandomForestRegressor(
    n_estimators=800,
    max_depth=10,
    min_samples_leaf=0.03,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cv_scores = cross_val_score(rf_model, X_train_export, y_train_export, cv=tscv, scoring="r2")
print(f"EXPORT RF Average CV R²: {np.mean(cv_scores):.3f}")

rf_model.fit(X_train_export, y_train_export)
y_pred_rf = rf_model.predict(X_test_export)

r2_rf = r2_score(y_test_export, y_pred_rf)
mae_rf = mean_absolute_error(y_test_export, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_export, y_pred_rf))

print("\nEXPORT Random Forest Results:")
print(f"R²: {r2_rf:.3f} | MAE: {mae_rf:.3f} | RMSE: {rmse_rf:.3f}")

# Plot
plt.figure(figsize=(10,5))
plt.plot(df_export_model.loc[df_export_model["Month"] > "2018-12", "Month"], y_test_export.values, "--", label="Actual")
plt.plot(df_export_model.loc[df_export_model["Month"] > "2018-12", "Month"], y_pred_rf, label="Predicted (RF)")
plt.title("EU Export Forecast – Random Forest")
plt.xlabel("Month"); plt.ylabel("log(Export Value)")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# =============== EXPORT FEATURE IMPORTANCE ===============
mod_rf_export = RandomForestRegressor(
    n_estimators=2200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
mod_rf_export.fit(X_train_export, y_train_export)

rf_importance_export = pd.Series(
    mod_rf_export.feature_importances_,
    index=X_train_export.columns
)

rf_importance_export_sorted = rf_importance_export.sort_values(ascending=True)

plt.figure(figsize=(8,6))
rf_importance_export_sorted.plot(kind="barh", color="seagreen")
plt.title("Random Forest Feature Importance – EXPORT Model")
plt.xlabel("Importance Score")
plt.ylabel("Predictor Variables")
plt.tight_layout()
plt.show()

print("\nTop 10 most important features for EXPORT model:")
print(rf_importance_export.sort_values(ascending=False).head(10))


####Import

In [ ]:
X_import = df_import_model[FEATURES_IMPORT]
y_import = df_import_model["log_OBS_VALUE"]
df_import_model["Month"] = pd.to_datetime(df_import_model["Month"])

X_train_import = X_import[df_import_model["Month"] <= "2018-12"]
X_test_import = X_import[df_import_model["Month"] > "2018-12"]
y_train_import = y_import[df_import_model["Month"] <= "2018-12"]
y_test_import = y_import[df_import_model["Month"] > "2018-12"]

# Tuned Random Forest
rf_model_import = RandomForestRegressor(
    n_estimators=800,
    max_depth=10,
    min_samples_leaf=0.03,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Time-series cross-validation
cv_scores_import = cross_val_score(rf_model_import, X_train_import, y_train_import, cv=tscv, scoring="r2")
print(f"IMPORT RF Average CV R²: {np.mean(cv_scores_import):.3f}")

# Train model
rf_model_import.fit(X_train_import, y_train_import)

# Predict on test set
y_pred_rf_import = rf_model_import.predict(X_test_import)

# Evaluate model performance
r2_rf_import = r2_score(y_test_import, y_pred_rf_import)
mae_rf_import = mean_absolute_error(y_test_import, y_pred_rf_import)
rmse_rf_import = np.sqrt(mean_squared_error(y_test_import, y_pred_rf_import))

print("\nIMPORT Random Forest Results:")
print(f"R²: {r2_rf_import:.3f} | MAE: {mae_rf_import:.3f} | RMSE: {rmse_rf_import:.3f}")

# Plot results
plt.figure(figsize=(10,5))
plt.plot(
    df_import_model.loc[df_import_model["Month"] > "2018-12", "Month"],
    y_test_import.values,
    "--",
    label="Actual"
)
plt.plot(
    df_import_model.loc[df_import_model["Month"] > "2018-12", "Month"],
    y_pred_rf_import,
    label="Predicted (RF)"
)
plt.title("EU Import Forecast – Random Forest")
plt.xlabel("Month")
plt.ylabel("log(Import Value)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============== IMPORT FEATURE IMPORTANCE ===============
mod_rf_import = RandomForestRegressor(
    n_estimators=2200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
mod_rf_import.fit(X_train_import, y_train_import)

rf_importance_import = pd.Series(
    mod_rf_import.feature_importances_,
    index=X_train_import.columns
)

rf_importance_import_sorted = rf_importance_import.sort_values(ascending=True)

plt.figure(figsize=(8,6))
rf_importance_import_sorted.plot(kind="barh", color="steelblue")
plt.title("Random Forest Feature Importance – IMPORT Model")
plt.xlabel("Importance Score")
plt.ylabel("Predictor Variables")
plt.tight_layout()
plt.show()

print("\nTop 10 most important features for IMPORT model:")
print(rf_importance_import.sort_values(ascending=False).head(10))

###5.XGBoost

####Export

In [ ]:
X_export = df_export_model[FEATURES_EXPORT]
y_export = df_export_model["log_OBS_VALUE"]

df_export_model["Month"] = pd.to_datetime(df_export_model["Month"])
X_train_export = X_export[df_export_model["Month"] <= "2018-12"]
X_test_export = X_export[df_export_model["Month"] > "2018-12"]
y_train_export = y_export[df_export_model["Month"] <= "2018-12"]
y_test_export = y_export[df_export_model["Month"] > "2018-12"]

xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.3,
    random_state=42,
    objective="reg:squarederror"
)

cv_scores = cross_val_score(xgb_model, X_train_export, y_train_export, cv=tscv, scoring="r2")
print(f"EXPORT XGBoost Average CV R²: {np.mean(cv_scores):.3f}")

xgb_model.fit(X_train_export, y_train_export)
y_pred_xgb = xgb_model.predict(X_test_export)

r2_xgb = r2_score(y_test_export, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test_export, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test_export, y_pred_xgb))

print("\nEXPORT XGBoost Results:")
print(f"R²: {r2_xgb:.3f} | MAE: {mae_xgb:.3f} | RMSE: {rmse_xgb:.3f}")

# Plot
plt.figure(figsize=(10,5))
plt.plot(df_export_model.loc[df_export_model["Month"] > "2018-12", "Month"], y_test_export.values, "--", label="Actual")
plt.plot(df_export_model.loc[df_export_model["Month"] > "2018-12", "Month"], y_pred_xgb, label="Predicted (XGBoost)")
plt.title("EU Export Forecast – XGBoost (2019–2025)")
plt.xlabel("Month"); plt.ylabel("log(Export Value)")
plt.legend(); plt.tight_layout(); plt.show()

####Import

In [ ]:
X_import = df_import_model[FEATURES_IMPORT]
y_import = df_import_model["log_OBS_VALUE"]

df_import_model["Month"] = pd.to_datetime(df_import_model["Month"])

# --- Time-based Train/Test Split ---
X_train_import = X_import[df_import_model["Month"] <= "2018-12"]
X_test_import = X_import[df_import_model["Month"] > "2018-12"]
y_train_import = y_import[df_import_model["Month"] <= "2018-12"]
y_test_import = y_import[df_import_model["Month"] > "2018-12"]

# --- Tuned XGBoost Model ---
xgb_model_import = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.3,
    random_state=42,
    objective="reg:squarederror"
)

# --- Time-Series Cross-Validation ---
cv_scores_import = cross_val_score(xgb_model_import, X_train_import, y_train_import, cv=tscv, scoring="r2")
print(f"IMPORT XGBoost Average CV R²: {np.mean(cv_scores_import):.3f}")

# --- Train Model ---
xgb_model_import.fit(X_train_import, y_train_import)

# --- Predict on Test Set ---
y_pred_xgb_import = xgb_model_import.predict(X_test_import)

# --- Evaluate Performance ---
r2_xgb_import = r2_score(y_test_import, y_pred_xgb_import)
mae_xgb_import = mean_absolute_error(y_test_import, y_pred_xgb_import)
rmse_xgb_import = np.sqrt(mean_squared_error(y_test_import, y_pred_xgb_import))

print("\nIMPORT XGBoost Results:")
print(f"R²: {r2_xgb_import:.3f} | MAE: {mae_xgb_import:.3f} | RMSE: {rmse_xgb_import:.3f}")

# --- Plot Actual vs Predicted ---
plt.figure(figsize=(10,5))
plt.plot(
    df_import_model.loc[df_import_model["Month"] > "2018-12", "Month"],
    y_test_import.values,
    "--",
    label="Actual"
)
plt.plot(
    df_import_model.loc[df_import_model["Month"] > "2018-12", "Month"],
    y_pred_xgb_import,
    label="Predicted (XGBoost)"
)
plt.title("EU Import Forecast – XGBoost (2019–2025)")
plt.xlabel("Month")
plt.ylabel("log(Import Value)")
plt.legend()
plt.tight_layout()
plt.show()

#Partial Dependence Plots for Random Forest and XGBoost

In [ ]:
# PARTIAL DEPENDENCE PLOTS (PDPs)
# For: Random Forest & XGBoost – Export and Import Models
from sklearn.inspection import PartialDependenceDisplay

# Choose key variables to visualize
key_features = ["GDP_EU", "Price_Index", "Price_change"]

# 1. Random Forest – EXPORT
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    rf_model,                  # trained Random Forest export model
    X_train_export,            # training features
    key_features,              # variables to visualize
    kind="average",
    ax=ax
)
plt.suptitle("Partial Dependence – Random Forest (Exports)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 2. Random Forest – IMPORT
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    rf_model_import,           # trained Random Forest import model
    X_train_import,            # training features
    key_features,
    kind="average",
    ax=ax
)
plt.suptitle("Partial Dependence – Random Forest (Imports)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- 3. XGBoost – EXPORT ---
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    xgb_model,                 # trained XGBoost export model
    X_train_export,            # training features
    key_features,
    kind="average",
    ax=ax
)
plt.suptitle("Partial Dependence – XGBoost (Exports)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
#  4. XGBoost – IMPORT
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    xgb_model_import,          # trained XGBoost import model
    X_train_import,            # training features
    key_features,
    kind="average",
    ax=ax
)
plt.suptitle("Partial Dependence – XGBoost (Imports)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()